<a href="https://www.kaggle.com/code/immadirohan/milestone-3-sentinelnet-ai?scriptVersionId=279024675" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

**Week - 5 : Anomaly Detection with Unsupervised Learning**

In [ ]:
# ===================================================================
# Network Intrusion Detection System - Complete Machine Learning Pipeline
# ===================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

# ===================================================================
# Step 1: Data Loading and Preprocessing
# ===================================================================

print("Step 1: Loading and preprocessing data...")

# Load dataset from Kaggle path
file_path = '/kaggle/input/wednesday-workinghours-pcap-iscx/Wednesday-workingHours.pcap_ISCX.csv'
df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

print(f"Original dataset shape: {df.shape}")

# Encode the target 'Label'
if 'Label' in df.columns:
    if df['Label'].dtype == 'object':
        le = LabelEncoder()
        df['Label_encoded'] = le.fit_transform(df['Label'])
        target_column = 'Label_encoded'
        print("Label encoded successfully")
    else:
        target_column = 'Label'
else:
    print("Error: 'Label' not found.")
    target_column = None

# ===================================================================
# Step 2: Correlation Analysis for Feature Selection
# ===================================================================

print("\nStep 2: Performing correlation analysis...")

# Select numeric columns
numeric_df_for_corr = df.select_dtypes(include=['int64', 'float64', 'float32']).copy()

# Add encoded label if needed
if target_column == 'Label_encoded' and target_column not in numeric_df_for_corr.columns:
    numeric_df_for_corr[target_column] = df[target_column]

if target_column:
    corr_matrix = numeric_df_for_corr.corr()
    target_corr = corr_matrix.abs()[target_column].sort_values(ascending=False)
    target_corr = target_corr.drop(labels=[target_column], errors='ignore')

    important_features = target_corr[target_corr > 0.3].index.tolist()
    non_important_features = target_corr[target_corr <= 0.3].index.tolist()

    print(f"Important Features (>0.3 correlation): {len(important_features)}")
    print("Top 10 important features:", important_features[:10])

    if len(important_features) == 0:
        print("No features > 0.3 found. Taking top 20.")
        important_features = target_corr.head(20).index.tolist()

    df_filtered = df[important_features + ['Label']].copy()
    print(f"Shape after correlation filtering: {df_filtered.shape}")
else:
    print("Cannot perform correlation analysis.")
    # Fallback: use all numeric features
    df_filtered = df.select_dtypes(include=['int64', 'float64', 'float32'])
    if 'Label' in df.columns:
        df_filtered['Label'] = df['Label']

# ===================================================================
# Step 3: PCA Analysis (Exploratory)
# ===================================================================

print("\nStep 3: Performing PCA analysis...")

X = df_filtered.drop(columns=['Label'])
y = df_filtered['Label']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Handle missing values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Perform PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print(f"Shape before PCA: {X_scaled.shape}")
print(f"Shape after PCA: {X_pca.shape}")

# Analyze explained variance
explained_variance = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(pca.explained_variance_))],
    'Explained_Variance': pca.explained_variance_ratio_
}).sort_values(by='Explained_Variance', ascending=False)

print("Top 10 PCA Components:")
print(explained_variance.head(10))

# ===================================================================
# Step 4: Random Forest Feature Importance
# ===================================================================

print("\nStep 4: Analyzing feature importance with Random Forest...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_feature_selector = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_feature_selector.fit(X_train, y_train)

importances = rf_feature_selector.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

top_10_features = feature_importance_df.head(10)
print("Top 10 Important Features:")
print(top_10_features)

# Create dataset with top 10 features
top_features = top_10_features['Feature'].tolist()
top_features_df = df_filtered[top_features + ['Label']]
top_features_df.to_csv("/kaggle/working/top10_features_dataset.csv", index=False)
print("Saved: top10_features_dataset.csv")

# ===================================================================
# Step 5: Model Training and Evaluation
# ===================================================================

print("\nStep 5: Training and evaluating models...")

# Load the top 10 features dataset
df_top10 = pd.read_csv("/kaggle/working/top10_features_dataset.csv")

X = df_top10.drop(columns=['Label'])
y = df_top10['Label']

print(f"Final dataset shape - X: {X.shape}, y: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features for models that need it
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train)
X_test_scaled = scaler_final.transform(X_test)

# ===================================================================
# Model 1: Random Forest Classifier
# ===================================================================

print("\n" + "="*50)
print("Random Forest Classifier")
print("="*50)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest Evaluation ===")
train_acc_rf = accuracy_score(y_train, rf.predict(X_train))
test_acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Training Accuracy: {train_acc_rf:.4f}")
print(f"Testing Accuracy: {test_acc_rf:.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, average='weighted', zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf, average='weighted', zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_rf, average='weighted', zero_division=0):.4f}")

# ===================================================================
# Model 2: Logistic Regression
# ===================================================================

print("\n" + "="*50)
print("Logistic Regression")
print("="*50)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("=== Logistic Regression Evaluation ===")
train_acc_lr = accuracy_score(y_train, lr.predict(X_train_scaled))
test_acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"Training Accuracy: {train_acc_lr:.4f}")
print(f"Testing Accuracy: {test_acc_lr:.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr, average='weighted', zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr, average='weighted', zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_lr, average='weighted', zero_division=0):.4f}")

# ===================================================================
# Model 3: Support Vector Machine (SVM)
# ===================================================================

print("\n" + "="*50)
print("Support Vector Machine (SVM)")
print("="*50)

svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)

print("=== SVM Evaluation ===")
train_acc_svm = accuracy_score(y_train, svm.predict(X_train_scaled))
test_acc_svm = accuracy_score(y_test, y_pred_svm)

print(f"Training Accuracy: {train_acc_svm:.4f}")
print(f"Testing Accuracy: {test_acc_svm:.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm, average='weighted', zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_svm, average='weighted', zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_svm, average='weighted', zero_division=0):.4f}")

# ===================================================================
# Step 6: Model Comparison Visualization
# ===================================================================

print("\nStep 6: Generating model comparison chart...")

# Prepare data for visualization
models_data = {
    'Model': ['Random Forest', 'Logistic Regression', 'SVM'],
    'Training Accuracy': [train_acc_rf * 100, train_acc_lr * 100, train_acc_svm * 100],
    'Testing Accuracy': [test_acc_rf * 100, test_acc_lr * 100, test_acc_svm * 100]
}

df_plot = pd.DataFrame(models_data)

# Create comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Training vs Testing Accuracy
x = np.arange(len(df_plot['Model']))
width = 0.35

ax1.bar(x - width/2, df_plot['Training Accuracy'], width, label='Training', alpha=0.7)
ax1.bar(x + width/2, df_plot['Testing Accuracy'], width, label='Testing', alpha=0.7)
ax1.set_xlabel('Model')
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Training vs Testing Accuracy')
ax1.set_xticks(x)
ax1.set_xticklabels(df_plot['Model'])
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Testing Accuracy Comparison
sns.barplot(x='Model', y='Testing Accuracy', data=df_plot, ax=ax2, palette='viridis')
ax2.set_title('Model Testing Accuracy Comparison')
ax2.set_ylabel('Testing Accuracy (%)')
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# ===================================================================
# Step 7: Save Models and Scaler
# ===================================================================

print("\nStep 7: Saving models and artifacts...")

# Save models
joblib.dump(rf, '/kaggle/working/random_forest_model.pkl')
joblib.dump(lr, '/kaggle/working/logistic_regression_model.pkl')
joblib.dump(svm, '/kaggle/working/svm_model.pkl')

# Save scaler
joblib.dump(scaler_final, '/kaggle/working/scaler.pkl')

# Save feature names
feature_names = {
    'top_features': top_features,
    'all_features': X.columns.tolist()
}
joblib.dump(feature_names, '/kaggle/working/feature_names.pkl')

print("Models and artifacts saved successfully!")
print("- Random Forest: /kaggle/working/random_forest_model.pkl")
print("- Logistic Regression: /kaggle/working/logistic_regression_model.pkl")
print("- SVM: /kaggle/working/svm_model.pkl")
print("- Scaler: /kaggle/working/scaler.pkl")
print("- Feature Names: /kaggle/working/feature_names.pkl")

# ===================================================================
# Step 8: Final Summary
# ===================================================================

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

summary_df = pd.DataFrame({
    'Model': ['Random Forest', 'Logistic Regression', 'SVM'],
    'Training Accuracy': [f"{train_acc_rf:.4f}", f"{train_acc_lr:.4f}", f"{train_acc_svm:.4f}"],
    'Testing Accuracy': [f"{test_acc_rf:.4f}", f"{test_acc_lr:.4f}", f"{test_acc_svm:.4f}"],
    'Precision': [
        f"{precision_score(y_test, y_pred_rf, average='weighted', zero_division=0):.4f}",
        f"{precision_score(y_test, y_pred_lr, average='weighted', zero_division=0):.4f}",
        f"{precision_score(y_test, y_pred_svm, average='weighted', zero_division=0):.4f}"
    ],
    'Recall': [
        f"{recall_score(y_test, y_pred_rf, average='weighted', zero_division=0):.4f}",
        f"{recall_score(y_test, y_pred_lr, average='weighted', zero_division=0):.4f}",
        f"{recall_score(y_test, y_pred_svm, average='weighted', zero_division=0):.4f}"
    ],
    'F1 Score': [
        f"{f1_score(y_test, y_pred_rf, average='weighted', zero_division=0):.4f}",
        f"{f1_score(y_test, y_pred_lr, average='weighted', zero_division=0):.4f}",
        f"{f1_score(y_test, y_pred_svm, average='weighted', zero_division=0):.4f}"
    ]
})

print(summary_df.to_string(index=False))

print(f"\nBest performing model (Testing Accuracy): {df_plot.loc[df_plot['Testing Accuracy'].idxmax(), 'Model']}")
print("Pipeline completed successfully!")

**Week 6: Model Evaluation and Fine-tuning**

In [ ]:
import os
import pandas as pd

# -----------------------------------
#  IF top10_features_dataset.csv 
# -----------------------------------

possible_paths = [
    "top10_features_dataset.csv",
    "/kaggle/working/top10_features_dataset.csv",
    "/kaggle/input/top10_features_dataset.csv"
]

csv_path = None
for path in possible_paths:
    if os.path.exists(path):
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError(
        "top10_features_dataset.csv not found. "
        "Run Milestone-2 first to generate the file."
    )

print("File found at:", csv_path)

df = pd.read_csv(csv_path)
print("Dataset loaded:", df.shape)
print(df.head())


# -------------------------------------------------------------
# STEP 1 — PCA Visualization
# -------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

X = df.drop(columns=['Label'])
y = df['Label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], s=5, alpha=0.5, color='blue')
plt.title("PCA Visualization")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()


# -------------------------------------------------------------
# STEP 2 — K-Means Anomaly Detection
# -------------------------------------------------------------
from sklearn.cluster import KMeans
import numpy as np

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(X_pca)

# Distance from each point to nearest cluster center
distances = np.min(
    np.linalg.norm(X_pca[:, None] - kmeans.cluster_centers_, axis=2),
    axis=1
)

threshold = np.percentile(distances, 98.5)
anomaly_kmeans = (distances > threshold).astype(int)

print("K-Means Anomaly Detection")
print("Normal count:", (anomaly_kmeans == 0).sum())
print("Anomaly count:", (anomaly_kmeans == 1).sum())

plt.figure(figsize=(10,6))
plt.scatter(X_pca[anomaly_kmeans==1, 0], X_pca[anomaly_kmeans==1, 1],
            c='red', label='Anomaly', s=15)
plt.scatter(X_pca[anomaly_kmeans==0, 0], X_pca[anomaly_kmeans==0, 1],
            c='blue', label='Normal', s=15)
plt.title("K-Means Anomaly Detection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True)
plt.show()


# -------------------------------------------------------------
# STEP 3 — Isolation Forest
# -------------------------------------------------------------
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(contamination=0.01, random_state=42)
iso_forest.fit(X_pca)

iso_preds = iso_forest.predict(X_pca)
anomaly_iso = (iso_preds == -1).astype(int)

print("Isolation Forest Results")
print("Normal count:", (anomaly_iso == 0).sum())
print("Anomaly count:", (anomaly_iso == 1).sum())

plt.figure(figsize=(10,6))
plt.scatter(X_pca[anomaly_iso==1, 0], X_pca[anomaly_iso==1, 1],
            c='red', label='Anomaly', s=15)
plt.scatter(X_pca[anomaly_iso==0, 0], X_pca[anomaly_iso==0, 1],
            c='blue', label='Normal', s=15)
plt.title("Isolation Forest Anomaly Detection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True)
plt.show()


# -------------------------------------------------------------
# WEEK 6 – MODEL COMPARISON PLOT
# -------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

models = ['Random Forest', 'SVM', 'Logistic Regression']
testing_accuracy = [0.9814, 0.9591, 0.9476]
precision = [0.9817, 0.9608, 0.9454]
recall = [0.9814, 0.9591, 0.9476]
f1 = [0.9810, 0.9583, 0.9431]

x = np.arange(len(models))
width = 0.2

plt.figure(figsize=(10,6))
plt.bar(x - 0.3, testing_accuracy, width, label='Accuracy')
plt.bar(x - 0.1, precision, width, label='Precision')
plt.bar(x + 0.1, recall, width, label='Recall')
plt.bar(x + 0.3, f1, width, label='F1 Score')

plt.xlabel('Models')
plt.ylabel('Scores')
plt.title('Model Performance Comparison')
plt.xticks(x, models)
plt.ylim(0.9, 1.0)
plt.legend()
plt.grid(axis='y', linestyle='--')
plt.show()


# -------------------------------------------------------------
# WEEK 6 – HYPERPARAMETER TUNING
# -------------------------------------------------------------
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import joblib

df = pd.read_csv(csv_path)
X = df.drop("Label", axis=1)
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf, rf_params, cv=3, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)

print("Best Random Forest Parameters:", rf_grid.best_params_)

joblib.dump(rf_grid.best_estimator_, "rf_model_tuned.pkl")
print("Saved tuned Random Forest model.")


**Step - 2 : Tune Hyperparameters and Cross Validation**


Hyperparameter tuning is the process of finding the best set of parameters that control the learning process of a model.
Instead of selecting them manually, techniques like Grid Search Cross-Validation (GridSearchCV) systematically test multiple parameter combinations.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# 1. LOAD ORIGINAL WEDNESDAY DATASET
# ------------------------------------------------------------

file_path = "/kaggle/input/wednesday-workinghours-pcap-iscx/Wednesday-workingHours.pcap_ISCX.csv"

df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()  # clean column names

print("Dataset loaded successfully!")
print(df.shape)


# ------------------------------------------------------------
# 2. ENCODE LABEL COLUMN
# ------------------------------------------------------------

if 'Label' not in df.columns:
    raise ValueError("ERROR: 'Label' column missing in dataset!")

le = LabelEncoder()
df["Label_encoded"] = le.fit_transform(df["Label"])


# ------------------------------------------------------------
# 3. CORRELATION ANALYSIS
# ------------------------------------------------------------

numeric_df = df.select_dtypes(include=["int64", "float64"]).copy()
numeric_df["Label_encoded"] = df["Label_encoded"]

corr_matrix = numeric_df.corr()
target_corr = corr_matrix["Label_encoded"].abs().sort_values(ascending=False)

# Remove the encoded label itself
target_corr = target_corr.drop("Label_encoded")

important_features = target_corr[target_corr > 0.3].index.tolist()
if len(important_features) == 0:
    important_features = target_corr.head(20).index.tolist()

print("Important correlated features:", important_features)


# ------------------------------------------------------------
# 4. RANDOM FOREST FEATURE IMPORTANCE
# ------------------------------------------------------------

df_filtered = df[important_features + ["Label"]].copy()

X = df_filtered.drop("Label", axis=1)
y = df_filtered["Label"]

# Handle missing values
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest model
rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Feature importance
fi = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

top10 = fi.head(10)["Feature"].tolist()
print("Top 10 Random Forest features:", top10)


# ------------------------------------------------------------
# 5. SAVE TOP 10 FEATURE DATASET
# ------------------------------------------------------------

top10_df = df[top10 + ["Label"]]
top10_df.to_csv("/kaggle/working/top10_features_dataset.csv", index=False)

print("File saved successfully: /kaggle/working/top10_features_dataset.csv")
print("Shape:", top10_df.shape)


In [ ]:
import pandas as pd

# -----------------------------------------
# Example: Create results_df before using it
# -----------------------------------------

results_list = [
    {"Model": "Logistic Regression", "Accuracy": 0.82},
    {"Model": "Random Forest", "Accuracy": 0.91},
    {"Model": "SVM", "Accuracy": 0.88},
    {"Model": "KNN", "Accuracy": 0.79},
]

# Convert list to DataFrame
results_df = pd.DataFrame(results_list)

# -----------------------------------------
# Now select the best model
# -----------------------------------------

best_model = results_df.loc[results_df['Accuracy'].idxmax()]

print("Best Model After Tuning:")
print(best_model)


**Step - 3 : Analyze Confusion Matrix and ROC Curve**

In [ ]:
#  Load required libraries
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score

#  Load the best model (change filename if your best model is different)
best_model = joblib.load("rf_model_tuned.pkl")

#  Make predictions
y_pred = best_model.predict(X_test_scaled)
y_prob = best_model.predict_proba(X_test_scaled)[:, 1] 

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix - Best Model")
plt.show()

**About ROC Curve**

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# Get the class labels
classes = np.unique(y_test)

# Binarize the true labels for One-vs-Rest ROC computation
y_test_bin = label_binarize(y_test, classes=classes)

# Get predicted probabilities (not class labels)
y_prob = best_model.predict_proba(X_test_scaled)

# Plot ROC Curve for each class
plt.figure(figsize=(8,6))
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Class {cls} (AUC = {roc_auc:.2f})')

# Diagonal reference line
plt.plot([0,1], [0,1], 'k--')

# Add labels and title
plt.title("ROC Curve - Multiclass Model (One-vs-Rest)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

The ROC curve shows how well the tuned Random Forest model can tell one class from another. It compares how often the model predicts a class correctly versus how often it gives false alarms.

If the curve is near the top-left corner and the AUC is close to 1.0, the model is performing very well. In our results, every class has an AUC above 0.99, which means the model is extremely accurate and can clearly separate the classes. This shows that the model learned well, the features are good, and the tuning was effective.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 1. Encode y_test (BENIGN → 0, MALICIOUS → 1)
le = LabelEncoder()
y_test_encoded = le.fit_transform(y_test)

# 2. K-Means clustering
kmeans = KMeans(n_clusters=len(np.unique(y_test_encoded)), random_state=42)
kmeans.fit(X_test)

y_kmeans_pred = kmeans.labels_

# 3. Map clusters → majority true label
labels = np.zeros_like(y_kmeans_pred)

for cluster_id in range(kmeans.n_clusters):
    cluster_mask = (y_kmeans_pred == cluster_id)

    if np.sum(cluster_mask) > 0:
        true_labels = y_test_encoded[cluster_mask]
        most_common = np.bincount(true_labels).argmax()
        labels[cluster_mask] = most_common

# 4. Compute accuracy
kmeans_acc = accuracy_score(y_test_encoded, labels)
print("K-Means Accuracy:", kmeans_acc)


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score

iso = IsolationForest(contamination=0.1, random_state=42)
iso.fit(X_train)

# Predict: 1 = normal, -1 = anomaly
y_pred_iso = iso.predict(X_test)
y_pred_iso = np.where(y_pred_iso == 1, 0, 1)

# Compute accuracy
iso_acc = accuracy_score(y_test, y_pred_iso)
print("Isolation Forest Accuracy:", iso_acc)
